# Day 3: Branches and Loops

Until now our graphs have been like a train on a single track. Start, go through every station in order, stop.

Real programs aren't like that. They make decisions. "If the payment failed, retry. If the user is a premium member, skip the ad. If the answer isn't good enough, try again."

Today the graph learns to do two things:

- **branch**: look at the state and pick which node to go to next
- **loop**: go back to an earlier node and repeat until some condition is met

This is honestly the heart of every AI agent you'll build later. An agent is basically a loop: think, act, check the result, think again. Once you understand today's notebook, agents won't feel mysterious.

Still no API key needed.

In [ ]:
%pip install -q -U langgraph

In [ ]:
import operator
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

In [ ]:
def show_graph(app):
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

## 1. A conditional edge is an `if` between nodes

Think of a railway junction. The train arrives, a signalman looks at where it's headed, and flips the switch to send it left or right.

In LangGraph, the signalman is called a **router**. It's a function that looks at the state and returns the **name of the next node**. You attach it with `add_conditional_edges`.

```
                 +--> even --+
  START --> check            +--> END
                 +--> odd  --+
```

In [ ]:
class NumberState(TypedDict):
    number: int
    result: str

In [ ]:
def check(state):
    # This node doesn't change anything. It's just the junction where the decision happens.
    return {}

In [ ]:
def even(state):
    return {"result": f"{state['number']} is even"}

In [ ]:
def odd(state):
    return {"result": f"{state['number']} is odd"}

In [ ]:
def even_or_odd(state) -> str:
    # The router. Notice it returns a node NAME, not a dictionary.
    if state["number"] % 2 == 0:
        return "even"
    return "odd"

In [ ]:
builder = StateGraph(NumberState)

In [ ]:
builder.add_node("check", check)
builder.add_node("even", even)
builder.add_node("odd", odd)

In [ ]:
builder.add_edge(START, "check")
# After "check", call even_or_odd to decide. The list tells LangGraph where it might go.
builder.add_conditional_edges("check", even_or_odd, ["even", "odd"])
builder.add_edge("even", END)
builder.add_edge("odd", END)

In [ ]:
number_app = builder.compile()
show_graph(number_app)

In the diagram, the dotted lines are the conditional edges. They mean "maybe this way, maybe that way".

In [ ]:
for n in (4, 7):
    print(number_app.invoke({"number": n})["result"])

A quick but important distinction:

- a **node** returns a *state update* (a dictionary)
- a **router** returns a *node name* (a string)

Routers should only decide. They shouldn't try to change the state. If you need to change something, do it in a node before the router.

## 2. Cleaner routers with `Literal`

Passing the list of possible destinations works, but there's a tidier way. Put the possible answers in the router's return type using `Literal`. Then LangGraph (and anyone reading your code) knows every place the router can send you.

Also, you can branch **straight from START**. You don't always need an empty "check" node.

In [ ]:
class Weather(TypedDict):
    temp: int
    advice: str

In [ ]:
def hot(state):
    return {"advice": "Drink water and stay indoors"}

In [ ]:
def mild(state):
    return {"advice": "Nice day for a walk"}

In [ ]:
def cold(state):
    return {"advice": "Take a jacket"}

In [ ]:
def by_temperature(state) -> Literal["hot", "mild", "cold"]:
    if state["temp"] >= 35:
        return "hot"
    if state["temp"] >= 18:
        return "mild"
    return "cold"

In [ ]:
builder = StateGraph(Weather)

In [ ]:
for fn in (hot, mild, cold):
    builder.add_node(fn.__name__, fn)
    builder.add_edge(fn.__name__, END)

In [ ]:
builder.add_conditional_edges(START, by_temperature)    # deciding right at the start

In [ ]:
weather_app = builder.compile()
show_graph(weather_app)

### Predict before you run

What advice do you get for 35, 18 and 17 degrees? Look carefully at the `>=` signs.

<details>
<summary>Click to see the answer</summary>

- 35 gives "Drink water and stay indoors" (35 >= 35 is true)
- 18 gives "Nice day for a walk" (18 >= 18 is true)
- 17 gives "Take a jacket"

Boundary values are exactly where routing bugs hide. Always test the edges.
</details>

In [ ]:
for t in (35, 18, 17):
    print(t, "degrees:", weather_app.invoke({"temp": t})["advice"])

### Your turn

Add a fourth branch called `freezing` for temperatures below 5. You'll need to add a node, add it to the `Literal`, and add a condition to the router.

In [ ]:
def freezing(state):
    return {"advice": "Stay home, it's freezing"}

In [ ]:
def by_temperature_v2(state) -> Literal["hot", "mild", "cold", "freezing"]:
    if state["temp"] >= 35:
        return "hot"
    if state["temp"] >= 18:
        return "mild"
    if state["temp"] >= 5:      # try changing this threshold
        return "cold"
    return "freezing"

In [ ]:
builder = StateGraph(Weather)

In [ ]:
for fn in (hot, mild, cold, freezing):
    builder.add_node(fn.__name__, fn)
    builder.add_edge(fn.__name__, END)

In [ ]:
builder.add_conditional_edges(START, by_temperature_v2)

In [ ]:
app = builder.compile()

In [ ]:
print(app.invoke({"temp": 2})["advice"])

## 3. When your router returns a label instead of a node name

Sometimes it's more natural for a router to answer "yes" or "no" rather than a node name. That's fine. You just pass a **mapping** from labels to nodes as the third argument.

In [ ]:
class Person(TypedDict):
    age: int
    status: str

In [ ]:
def check_age(state):
    return {}

In [ ]:
def allow(state):
    return {"status": "Welcome in!"}

In [ ]:
def deny(state):
    return {"status": "Sorry, 18+ only"}

In [ ]:
def is_adult(state) -> str:
    return "yes" if state["age"] >= 18 else "no"

In [ ]:
builder = StateGraph(Person)

In [ ]:
builder.add_node("check_age", check_age)
builder.add_node("allow", allow)
builder.add_node("deny", deny)

In [ ]:
builder.add_edge(START, "check_age")
# "yes" goes to allow, "no" goes to deny
builder.add_conditional_edges("check_age", is_adult, {"yes": "allow", "no": "deny"})
builder.add_edge("allow", END)
builder.add_edge("deny", END)

In [ ]:
age_app = builder.compile()

In [ ]:
print(age_app.invoke({"age": 21})["status"])
print(age_app.invoke({"age": 15})["status"])

## 4. Loops: going back to an earlier node

Here's where it gets interesting. A conditional edge can point **backwards**, to a node that already ran. That creates a loop.

Think of a board game where you keep rolling the dice until you get a six. Roll, check, roll again, check again, and only stop when you see a six.

```
  START --> roll_dice --> got a six? --no--+
                ^                          |
                +--------------------------+
                          |
                         yes
                          v
                         END
```

To keep the result the same every time you run this notebook, we'll use a fixed list of "dice rolls" instead of random numbers.

In [ ]:
class DiceState(TypedDict):
    planned_rolls: list[int]                     # our pretend dice
    history: Annotated[list[int], operator.add]  # every roll gets added here

In [ ]:
def roll_dice(state):
    # Pick the next roll from the list, based on how many we've already done
    roll = state["planned_rolls"][len(state["history"])]
    print("rolled", roll)
    return {"history": [roll]}

In [ ]:
def got_six(state) -> Literal["roll_dice", "__end__"]:
    if state["history"][-1] == 6:
        return END            # END is actually just the string "__end__"
    return "roll_dice"        # go back and roll again

In [ ]:
builder = StateGraph(DiceState)

In [ ]:
builder.add_node("roll_dice", roll_dice)

In [ ]:
builder.add_edge(START, "roll_dice")
builder.add_conditional_edges("roll_dice", got_six)

In [ ]:
dice_app = builder.compile()
show_graph(dice_app)

### Predict before you run

The planned rolls are `[2, 5, 1, 6, 3]`. How many times does `roll_dice` run, and what's in `history` at the end?

<details>
<summary>Click to see the answer</summary>

Four times. `history` ends up as `[2, 5, 1, 6]`.

The 3 is never rolled, because the loop stops as soon as it sees the 6.
</details>

In [ ]:
result = dice_app.invoke({"planned_rolls": [2, 5, 1, 6, 3], "history": []})
print("history:", result["history"])

## 5. The safety net: recursion limit

Now a slightly scary question. What if the exit condition is never met? What if the six never comes?

Without protection, your program would loop forever (and if each step called a paid LLM, your bill would loop forever too). LangGraph protects you: by default it stops after **25 steps** and raises a `GraphRecursionError`. You can set your own limit through `config`.

In [ ]:
from langgraph.errors import GraphRecursionError

In [ ]:
try:
    # Fifty 1s before any 6, but we only allow 5 steps
    dice_app.invoke({"planned_rolls": [1] * 50 + [6], "history": []},
                    config={"recursion_limit": 5})
except GraphRecursionError:
    print("Stopped by the recursion limit. The loop needs a better way out.")

In real projects, don't rely only on this limit. Add your own counter to the state, like `attempts`, and stop after, say, 3 tries. The recursion limit is the seatbelt, not the brakes.

## 6. `Command`: update the state and pick the next node in one go

So far, a node updates the state and then a separate router decides where to go. Sometimes it feels more natural for the node that did the work to also decide the next step. Think of a receptionist who checks your ID **and** points you to the right room, both in one go.

For that, a node can return a `Command`:

```python
return Command(update={...}, goto="next_node")
```

Put the possible destinations in the return type, `Command[Literal[...]]`, so LangGraph knows the options. Then you don't need `add_conditional_edges` at all.

In [ ]:
from langgraph.types import Command

In [ ]:
class LoginState(TypedDict):
    password: str
    message: str

In [ ]:
def check_password(state) -> Command[Literal["welcome", "reject"]]:
    if state["password"] == "langgraph123":
        return Command(update={"message": "password ok"}, goto="welcome")
    return Command(update={"message": "wrong password"}, goto="reject")

In [ ]:
def welcome(state):
    return {"message": state["message"] + ", logged in"}

In [ ]:
def reject(state):
    return {"message": state["message"] + ", please try again"}

In [ ]:
builder = StateGraph(LoginState)

In [ ]:
builder.add_node("check_password", check_password)
builder.add_node("welcome", welcome)
builder.add_node("reject", reject)

In [ ]:
builder.add_edge(START, "check_password")    # note: no conditional edges needed
builder.add_edge("welcome", END)
builder.add_edge("reject", END)

In [ ]:
login_app = builder.compile()
show_graph(login_app)

In [ ]:
print(login_app.invoke({"password": "langgraph123"})["message"])
print(login_app.invoke({"password": "1234"})["message"])

Which one should you use? Use a separate router when the decision logic is something you want to read and test on its own. Use `Command` when the node doing the work naturally knows where to go next. You'll see `Command` a lot when we get to multi-agent systems on Day 12.

## 7. Common mistakes

**Mistake 1: The router returns a name that doesn't exist.**
A typo in a node name is a very common one, and it's nasty because there's **no error**. LangGraph prints a small warning, ignores the bad route, and the graph just... stops. You get a result with your answer missing.

In [ ]:
def typo_router(state) -> str:
    return "evem"            # meant to say "even"

In [ ]:
builder = StateGraph(NumberState)

In [ ]:
builder.add_node("even", even)
builder.add_node("odd", odd)

In [ ]:
builder.add_conditional_edges(START, typo_router)
builder.add_edge("even", END)
builder.add_edge("odd", END)

In [ ]:
app = builder.compile()

In [ ]:
print(app.invoke({"number": 2}))
# No "result" key. Nothing ran. Scroll up a little and you may see the warning about "evem".

**Fix:** use `Literal[...]` return types (or the list of destinations). Your editor can then flag names that don't match, and when a key you expected is missing from the result, check your router's spelling first.

**Mistake 2: Returning a dictionary from a router.**
Routers return a name. If you return `{"next": "even"}`, LangGraph doesn't know what to do with it. If you want to update state *and* route, use `Command`.

**Mistake 3: A loop whose exit condition can never be true.**
For example, checking `state["count"] == 10` while the count goes up by 3 (3, 6, 9, 12...). It skips right past 10. Prefer `>=` over `==` in exit conditions, and keep an attempts counter.

## 8. Mini project: support ticket triage

Let's combine everything. A customer message comes in. We classify it, set a priority, send it to the right team, and finalize the reply.

```
  START -> classify -> set_priority --+--> billing --+
                                      +--> tech -----+--> finalize -> END
                                      +--> general --+
```

In [ ]:
class Ticket(TypedDict):
    message: str
    category: str
    priority: str
    reply: str
    log: Annotated[list[str], operator.add]

In [ ]:
def classify(state):
    text = state["message"].lower()
    if any(word in text for word in ("refund", "charge", "invoice")):
        category = "billing"
    elif any(word in text for word in ("error", "crash", "bug")):
        category = "tech"
    else:
        category = "general"
    return {"category": category, "log": [f"category={category}"]}

In [ ]:
def set_priority(state):
    urgent = any(word in state["message"].lower() for word in ("urgent", "asap"))
    priority = "high" if urgent else "normal"
    return {"priority": priority, "log": [f"priority={priority}"]}

In [ ]:
def route_to_team(state) -> Literal["billing", "tech", "general"]:
    # The category names were chosen to match the node names, so this is easy
    return state["category"]

In [ ]:
def billing(state):
    return {"reply": "Our billing team is checking your payment.", "log": ["billing"]}

In [ ]:
def tech(state):
    return {"reply": "Our tech team is looking into the problem.", "log": ["tech"]}

In [ ]:
def general(state):
    return {"reply": "Thanks for reaching out, we'll reply soon.", "log": ["general"]}

In [ ]:
def finalize(state):
    prefix = "[HIGH PRIORITY] " if state["priority"] == "high" else ""
    return {"reply": prefix + state["reply"], "log": ["finalized"]}

In [ ]:
builder = StateGraph(Ticket)

In [ ]:
for fn in (classify, set_priority, billing, tech, general, finalize):
    builder.add_node(fn.__name__, fn)

In [ ]:
builder.add_edge(START, "classify")
builder.add_edge("classify", "set_priority")
builder.add_conditional_edges("set_priority", route_to_team)
for team in ("billing", "tech", "general"):
    builder.add_edge(team, "finalize")
builder.add_edge("finalize", END)

In [ ]:
triage = builder.compile()
show_graph(triage)

### Predict before you run

The message is "URGENT: app crashes with an error!". Which team handles it, what's the priority, and what will `log` look like?

<details>
<summary>Click to see the answer</summary>

Team: tech (because of "crash" and "error"). Priority: high (because of "urgent", after lowercasing).

`log` will be `['category=tech', 'priority=high', 'tech', 'finalized']`, one entry per node, in the order they ran.
</details>

In [ ]:
result = triage.invoke({"message": "URGENT: app crashes with an error!", "log": []})
for key, value in result.items():
    print(f"{key:9}: {value}")

### Your turn

Send a few messages of your own through `triage`. Try one that mentions both "refund" and "error". Which team wins, and why?

In [ ]:
my_message = "I need a refund, the app showed an error"     # change this
print(triage.invoke({"message": my_message, "log": []})["reply"])

(It goes to billing, because `classify` checks the billing words first. Order of `if` checks matters.)

## 9. Practice

**Exercise 1.** Grade a student from `marks`: 75 or more is "distinction", 40 or more is "pass", anything else is "fail". Route straight from START with a `Literal` router.

**Exercise 2 (a loop).** Start with a number `n` and keep halving it (integer division) until it's 1 or less. Store every new value in `steps`. For `n = 40` you should get `[20, 10, 5, 2, 1]`.

In [ ]:
# Solution 1
class Marks(TypedDict):
    marks: int
    grade: str

In [ ]:
def distinction(state):
    return {"grade": "distinction"}

In [ ]:
def passed(state):
    return {"grade": "pass"}

In [ ]:
def fail(state):
    return {"grade": "fail"}

In [ ]:
def grade_router(state) -> Literal["distinction", "passed", "fail"]:
    if state["marks"] >= 75:
        return "distinction"
    if state["marks"] >= 40:
        return "passed"
    return "fail"

In [ ]:
builder = StateGraph(Marks)

In [ ]:
for fn in (distinction, passed, fail):
    builder.add_node(fn.__name__, fn)
    builder.add_edge(fn.__name__, END)

In [ ]:
builder.add_conditional_edges(START, grade_router)

In [ ]:
grade_app = builder.compile()

In [ ]:
print([grade_app.invoke({"marks": m})["grade"] for m in (90, 50, 20)])

In [ ]:
# Solution 2
class Halving(TypedDict):
    n: int
    steps: Annotated[list[int], operator.add]

In [ ]:
def halve(state):
    new_value = state["n"] // 2
    return {"n": new_value, "steps": [new_value]}

In [ ]:
def keep_going(state) -> Literal["halve", "__end__"]:
    return END if state["n"] <= 1 else "halve"

In [ ]:
builder = StateGraph(Halving)

In [ ]:
builder.add_node("halve", halve)

In [ ]:
builder.add_edge(START, "halve")
builder.add_conditional_edges("halve", keep_going)

In [ ]:
app = builder.compile()

In [ ]:
result = app.invoke({"n": 40, "steps": []})
print(result)
assert result["steps"] == [20, 10, 5, 2, 1]

## Wrapping up

Today the graph learned to think a little. A **router** looks at the state and returns the next node's name. `Literal` types make the options clear. Pointing an edge backwards gives you a **loop**, and the **recursion limit** stops runaway loops. `Command` lets one node update the state and choose the next step together.

Here's something worth noticing. You now know the entire core of LangGraph. Everything from here on, chatbots, tools, agents, memory, multi-agent teams, is built from exactly these pieces. We just start putting an LLM inside some of the nodes.

Tomorrow we do exactly that. We'll connect OpenAI safely using a `.env` file and learn how chat messages work.